In [0]:
from pyspark.sql.functions import trim, col, StringType
import pyspark.sql.functions as F


df = spark.table("dev_project.bronze.erp_cust_az12")

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


df = df.withColumn(
    "GEN",
    F.when(F.upper(F.col("GEN")).startswith("M"), "Male")
     .when(F.upper(F.col("GEN")).startswith("F"), "Female")
     .otherwise("n/a")
)

df = df.withColumn(
    "BDATE",
    F.when(F.col("BDATE") >  F.current_date(), None).otherwise(F.col("BDATE"))
)
name_map = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}

for old_name,new_name in name_map.items():
    df = df.withColumnRenamed(old_name,new_name)


df.display()


df.write.mode("overwrite").saveAsTable("dev_project.silver.erp_cust_az12")


In [0]:
import pyspark.sql.functions as F

df.filter(F.col("birth_date") > F.current_date()).count()